# Robustness analysis

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
from transformers import AutoModelForSeq2SeqLM
nltk.download("punkt")
nltk.download("stopwords")
import spacy

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawanjkumar/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)

# dataset = dataset[9500:]   # test split


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawanjkumar/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawanjkumar/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()
        
def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)
    
import random

def add_noise(sentences, roles, drop_prob=0.1):
    new_sent, new_roles = [], []
    for s, r in zip(sentences, roles):
        if np.random.rand() > drop_prob:
            new_sent.append(s)
            new_roles.append(r)
    return new_sent, new_roles


def shuffle_input(sentences, roles):
    combined = list(zip(sentences, roles))
    random.shuffle(combined)
    s, r = zip(*combined)
    return list(s), list(r)


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

# SummaC
# summac_conv = SummaCConv(device=device)
# summac_zs = SummaCZS(device=device)
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)


import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import json
import math
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ==============================
# Legal Role Order
# ==============================

LEGAL_FLOW = [
    "PREAMBLE", "FAC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT",
    "RLC", "STA", "PRE_RELIED","PRE_NOT_RELIED", "ANALYSIS", "RATIO", "RPC"
]

# ==============================
# Role Importance Weights
# ==============================

ROLE_WEIGHTS = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PREAMBLE": 0.4,
    "NONE": 0.2
}

# ==============================
# Position Weight
# ==============================

def position_weight(idx, total):
    """
    Early and late sentences get higher weight.
    """
    if total <= 1:
        return 1.0
    relative = idx / total
    if relative < 0.15:
        return 1.2
    elif relative > 0.85:
        return 1.15
    else:
        return 1.0

def mmr_selection(sentences, scores, embeddings, top_k, lambda_param=0.7):
    selected = []
    candidates = list(range(len(sentences)))
    while len(selected) < min(top_k, len(sentences)):
        mmr_scores = []
        for idx in candidates:
            relevance = scores[idx]
            redundancy = 0
            if selected:
                sim = cosine_similarity(
                    embeddings[idx].reshape(1, -1),
                    embeddings[selected]
                )[0]
                redundancy = max(sim)
            mmr_score = lambda_param * relevance - (1 - lambda_param) * redundancy
            mmr_scores.append((mmr_score, idx))
        mmr_scores.sort(reverse=True)
        best_idx = mmr_scores[0][1]
        selected.append(best_idx)
        candidates.remove(best_idx)
    return selected

def compute_hybrid_scores(sentences):
    # TF-IDF
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(sentences)
    tfidf_embeddings = tfidf_matrix.toarray()
    tfidf_scores = np.sum(tfidf_embeddings, axis=1)

    # SBERT
    sbert_embeddings = sbert_model.encode(sentences)
    doc_emb = np.mean(sbert_embeddings, axis=0)
    semantic_scores = cosine_similarity(
        sbert_embeddings,
        doc_emb.reshape(1, -1)
    ).flatten()

    # Normalize
    tfidf_scores = tfidf_scores / (np.max(tfidf_scores) + 1e-8)
    semantic_scores = semantic_scores / (np.max(semantic_scores) + 1e-8)
    # Hybrid
    hybrid_scores = 0.65 * tfidf_scores + 0.35 * semantic_scores
    return hybrid_scores, tfidf_embeddings
    
def global_mmr(selected_indices, embeddings):
    final = []

    for idx in selected_indices:
        if not final:
            final.append(idx)
            continue

        sim = cosine_similarity(
            embeddings[idx].reshape(1, -1),
            embeddings[final]
        )[0]

        if max(sim) < 0.85:
            final.append(idx)

    return final  
def ralss_sentence_selection(sentences, roles, config, compression_ratio=0.25):

    if len(sentences) == 0:
        return []

    summary_limit = estimate_summary_length(sentences)

    # Hybrid scoring
    base_scores, embeddings = compute_hybrid_scores(sentences)

    weighted_scores = []
    for i in range(len(sentences)):

        role = roles[i]

        # A1: Role Weights Ablation
        role_weight = ROLE_WEIGHTS.get(role, 0.2) if config["use_role_weights"] else 1.0

        #  A2: Position Weights Ablation
        pos_weight = position_weight(i, len(sentences)) if config["use_position_weights"] else 1.0

        length_bonus = 1 + (len(sentences[i].split()) / 100)

        score = base_scores[i] * role_weight * pos_weight * length_bonus
        weighted_scores.append(score)

    weighted_scores = np.array(weighted_scores)

    # Group by role
    role_groups = {}
    for i, role in enumerate(roles):
        role_groups.setdefault(role, []).append(i)

    selected_indices = []

    # --------------------------
    # Selection
    # --------------------------
    for role, idxs in role_groups.items():

        #  A4: Dynamic Budget Ablation
        if config["use_dynamic_budget"]:
            role_weight = ROLE_WEIGHTS.get(role, 0.2)
            role_budget = math.ceil(len(idxs) * role_weight * compression_ratio)
        else:
            role_budget = max(1, int(len(sentences) * compression_ratio / len(role_groups)))

        role_budget = max(role_budget, 1)

        role_sentences = [sentences[i] for i in idxs]
        role_scores = [weighted_scores[i] for i in idxs]
        role_embeddings = embeddings[idxs]

        #  A3: MMR Ablation
        if config["use_mmr"]:
            chosen = mmr_selection(
                role_sentences,
                role_scores,
                role_embeddings,
                role_budget
            )
        else:
            chosen = sorted(
                range(len(role_scores)),
                key=lambda x: role_scores[x],
                reverse=True
            )[:role_budget]

        selected_indices.extend([idxs[i] for i in chosen])

    #  Global MMR Ablation
    if config["use_global_mmr"]:
        selected_indices = global_mmr(selected_indices, embeddings)

    # Limit size
    if len(selected_indices) > summary_limit:
        ranked = sorted(
            [(weighted_scores[i], i) for i in selected_indices],
            reverse=True
        )
        selected_indices = [i for _, i in ranked[:summary_limit]]

    #  A5: Legal Order Ablation
    if config["use_legal_order"]:
        role_order = {r: i for i, r in enumerate(LEGAL_FLOW)}
        selected_indices.sort(
            key=lambda x: (role_order.get(roles[x], 999), x)
        )

    return selected_indices

def build_structured_input(sentences, roles, selected_indices):
    structured = []

    for idx in selected_indices:
        role = roles[idx]
        sent = sentences[idx]
        structured.append(f"[{role}] {sent}")

    return " ".join(structured)
    
from transformers import pipeline

def evaluate_metrics(pred, reference, judgement_chunks):
    
    r = rouge.compute(predictions=[pred], references=[reference])
    bleu_score = bleu.compute(predictions=[pred], references=[[reference]])["bleu"]
    meteor_score = meteor.compute(predictions=[pred], references=[reference])["meteor"]
    
    bert = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )["f1"][0]
    
    fact = factcc_chunked_fast(pred, judgement_chunks)

    return {
        "rouge1": r["rouge1"],
        "rouge2": r["rouge2"],
        "rougel": r["rougeL"],
        "bleu": bleu_score,
        "meteor": meteor_score,
        "bert": bert,
        "factcc": fact
    }
    
def percentage_drop(original, perturbed):
    if original == 0:
        return 0
    return ((original - perturbed) / original) * 100
    
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from nltk import sent_tokenize
import gc  

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks

def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary

results = {}

robustness_results = {
    "noise": {"rouge1": [],"rouge2": [],"rougel": [], "bleu": [], "meteor": [], "bert": [], "factcc": []},
    "shuffle": {"rouge1": [],"rouge2": [],"rougel": [], "bleu": [], "meteor": [], "bert": [], "factcc": []}
}
MAIN_CONFIG = {
    "use_role_weights": False,
    "use_position_weights": True,
    "use_mmr": False,
    "use_dynamic_budget": True,
    "use_legal_order": True,
    "use_global_mmr": False
}
for doc in tqdm(dataset, desc="Robustness Analysis"):

    reference = " ".join(doc["headnote_sent"])

    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]

    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    # =========================
    #  ORIGINAL
    # =========================
    selected = ralss_sentence_selection(sentences, roles, MAIN_CONFIG)
    ranked_text = build_structured_input(sentences, roles, selected)
    pred = generate_summary_chunked(ranked_text, tokenizer, model)
    original_scores = evaluate_metrics(pred, reference, judgement_chunks)

    # =========================
    #  NOISE TEST
    # =========================
    noisy_sent, noisy_roles = add_noise(sentences, roles, drop_prob=0.1)

    if len(noisy_sent) > 5:
        selected_noise = ralss_sentence_selection(
            noisy_sent,
            noisy_roles,
            MAIN_CONFIG
        )

        # FIX: use noisy roles + selected_noise
        ranked_text_noise = build_structured_input(
            noisy_sent,
            noisy_roles,
            selected_noise
        )

        pred_noise = generate_summary_chunked(
            ranked_text_noise,
            tokenizer,
            model
        )

        noise_scores = evaluate_metrics(
            pred_noise,
            reference,
            judgement_chunks
        )

        for key in robustness_results["noise"]:
            drop = percentage_drop(original_scores[key], noise_scores[key])
            robustness_results["noise"][key].append(drop)

    # =========================
    #  SHUFFLE TEST
    # =========================
    shuf_sent, shuf_roles = shuffle_input(sentences, roles)

    selected_shuffle = ralss_sentence_selection(
        shuf_sent,
        shuf_roles,
        MAIN_CONFIG
    )

    # FIX: use shuffled roles + selected_shuffle
    ranked_text_shuffle = build_structured_input(
        shuf_sent,
        shuf_roles,
        selected_shuffle
    )

    pred_shuffle = generate_summary_chunked(
        ranked_text_shuffle,
        tokenizer,
        model
    )

    shuffle_scores = evaluate_metrics(
        pred_shuffle,
        reference,
        judgement_chunks
    )

    for key in robustness_results["shuffle"]:
        drop = percentage_drop(original_scores[key], shuffle_scores[key])
        robustness_results["shuffle"][key].append(drop)

    # =========================
    # CLEANUP
    # =========================
    torch.cuda.empty_cache()
    gc.collect()
    
print("\n ROBUSTNESS RESULTS (Average % Drop)\n")

for test_type in robustness_results:
    print(f"--- {test_type.upper()} ---")
    for metric in robustness_results[test_type]:
        avg_drop = np.mean(robustness_results[test_type][metric])
        print(f"{metric.upper()} Drop {avg_drop:.2f}%")
    print()

In [ ]:
# ==========================================
# ROBUSTNESS RESULT STORAGE
# ==========================================
def perturb_roles(roles, perturb_prob=0.15):

    possible_roles = list(ROLE_WEIGHTS.keys())

    new_roles = []

    for r in roles:

        if np.random.rand() < perturb_prob:
            candidate = random.choice(possible_roles)

            while candidate == r:
                candidate = random.choice(possible_roles)

            new_roles.append(candidate)

        else:
            new_roles.append(r)

    return new_roles

IRRELEVANT_SENTENCES = [
    "The matter was listed before the court.",
    "The learned counsel appeared before the bench.",
    "The case was heard on multiple dates.",
    "Administrative details were recorded.",
    "The registry completed procedural formalities."
]

def inject_irrelevant_sentences(sentences, roles, inject_ratio=0.1):

    new_sent = sentences.copy()
    new_roles = roles.copy()

    n_inject = max(1, int(len(sentences) * inject_ratio))

    for _ in range(n_inject):

        idx = random.randint(0, len(new_sent))

        sent = random.choice(IRRELEVANT_SENTENCES)

        new_sent.insert(idx, sent)
        new_roles.insert(idx, "NONE")

    return new_sent, new_roles


robustness_results = {

    "role_noise": {
        "rouge1": [],
        "rouge2": [],
        "rougel": [],
        "bleu": [],
        "meteor": [],
        "bert": [],
        "factcc": []
    },

    "irrelevant": {
        "rouge1": [],
        "rouge2": [],
        "rougel": [],
        "bleu": [],
        "meteor": [],
        "bert": [],
        "factcc": []
    }
}

import gc

for doc in tqdm(dataset, desc="Robustness Analysis"):

    reference = " ".join(doc["headnote_sent"])

    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]

    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    # =====================================
    # ORIGINAL PREDICTION
    # =====================================

    selected = ralss_sentence_selection(
        sentences,
        roles,
        MAIN_CONFIG
    )

    ranked_text = build_structured_input(
        sentences,
        roles,
        selected
    )

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    original_scores = evaluate_metrics(
        pred,
        reference,
        judgement_chunks
    )

    # =====================================
    # ROLE PERTURBATION TEST
    # =====================================

    perturbed_roles = perturb_roles(
        roles,
        perturb_prob=0.15
    )

    selected_role_noise = ralss_sentence_selection(
        sentences,
        perturbed_roles,
        MAIN_CONFIG
    )

    ranked_text_role = build_structured_input(
        sentences,
        perturbed_roles,
        selected_role_noise
    )

    pred_role = generate_summary_chunked(
        ranked_text_role,
        tokenizer,
        model
    )

    role_scores = evaluate_metrics(
        pred_role,
        reference,
        judgement_chunks
    )

    for key in robustness_results["role_noise"]:

        drop = percentage_drop(
            original_scores[key],
            role_scores[key]
        )

        robustness_results["role_noise"][key].append(drop)

    # =====================================
    #  IRRELEVANT SENTENCE INJECTION
    # =====================================

    irr_sent, irr_roles = inject_irrelevant_sentences(
        sentences,
        roles,
        inject_ratio=0.10
    )

    selected_irr = ralss_sentence_selection(
        irr_sent,
        irr_roles,
        MAIN_CONFIG
    )

    ranked_text_irr = build_structured_input(
        irr_sent,
        irr_roles,
        selected_irr
    )

    pred_irr = generate_summary_chunked(
        ranked_text_irr,
        tokenizer,
        model
    )

    irr_scores = evaluate_metrics(
        pred_irr,
        reference,
        judgement_chunks
    )

    for key in robustness_results["irrelevant"]:

        drop = percentage_drop(
            original_scores[key],
            irr_scores[key]
        )

        robustness_results["irrelevant"][key].append(drop)

    # =====================================
    # CLEANUP
    # =====================================

    torch.cuda.empty_cache()
    gc.collect()

print("\n ROBUSTNESS RESULTS (Average % Drop)\n")

for test_type in robustness_results:

    print(f"--- {test_type.upper()} ---")

    for metric in robustness_results[test_type]:

        avg_drop = np.mean(
            robustness_results[test_type][metric]
        )

        print(
            f"{metric.upper()} Drop  {avg_drop:.2f}%"
        )

    print()

In [ ]:
#  ROBUSTNESS RESULTS (Average % Drop)

# --- NOISE ---
# ROUGE1 Drop  0.98%
# ROUGE2 Drop  1.06%
# ROUGEL Drop  0.12%
# BLEU Drop  2.61%
# METEOR Drop  3.22%
# BERT Drop  0.06%
# FACTCC Drop  0.04%

# --- SHUFFLE ---
# ROUGE1 Drop  1.00%
# ROUGE2 Drop  2.29%
# ROUGEL Drop  4.70%
# BLEU Drop  2.42%
# METEOR Drop  1.56%
# BERT Drop  0.33%
# FACTCC Drop  0.04%

# --- ROLE_NOISE ---
# ROUGE1 Drop  -0.17%
# ROUGE2 Drop  -0.44%
# ROUGEL Drop  0.66%
# BLEU Drop  -1.31%
# METEOR Drop  -0.64%
# BERT Drop  0.11%
# FACTCC Drop  0.22%

# --- IRRELEVANT ---
# ROUGE1 Drop  -0.71%
# ROUGE2 Drop  -1.27%
# ROUGEL Drop  -0.63%
# BLEU Drop  -3.67%
# METEOR Drop  -1.94%
# BERT Drop  -0.02%
# FACTCC Drop  0.07%

In [ ]:
def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()



# ==============================
# Legal Role Order
# ==============================

LEGAL_FLOW = [
    "PREAMBLE", "FAC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT",
    "RLC", "STA", "PRE_RELIED","PRE_NOT_RELIED", "ANALYSIS", "RATIO", "RPC"
]

# ==============================
# Role Importance Weights
# ==============================

ROLE_WEIGHTS = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PREAMBLE": 0.4,
    "NONE": 0.2
}

# ==============================
# Position Weight
# ==============================

def position_weight(idx, total):
    """
    Early and late sentences get higher weight.
    """
    if total <= 1:
        return 1.0
    relative = idx / total
    if relative < 0.15:
        return 1.2
    elif relative > 0.85:
        return 1.15
    else:
        return 1.0

def mmr_selection(sentences, scores, embeddings, top_k, lambda_param=0.7):
    selected = []
    candidates = list(range(len(sentences)))
    while len(selected) < min(top_k, len(sentences)):
        mmr_scores = []
        for idx in candidates:
            relevance = scores[idx]
            redundancy = 0
            if selected:
                sim = cosine_similarity(
                    embeddings[idx].reshape(1, -1),
                    embeddings[selected]
                )[0]
                redundancy = max(sim)
            mmr_score = lambda_param * relevance - (1 - lambda_param) * redundancy
            mmr_scores.append((mmr_score, idx))
        mmr_scores.sort(reverse=True)
        best_idx = mmr_scores[0][1]
        selected.append(best_idx)
        candidates.remove(best_idx)
    return selected

def compute_hybrid_scores(sentences, config):

    # TF-IDF
    if config["use_tfidf"]:
        vectorizer = TfidfVectorizer(stop_words="english")
        tfidf_matrix = vectorizer.fit_transform(sentences)
        tfidf_embeddings = tfidf_matrix.toarray()
        tfidf_scores = np.sum(tfidf_embeddings, axis=1)
        tfidf_scores = tfidf_scores / (np.max(tfidf_scores) + 1e-8)
    else:
        tfidf_embeddings = None
        tfidf_scores = np.zeros(len(sentences))

    # SBERT
    if config["use_sbert"]:
        sbert_embeddings = sbert_model.encode(sentences)
        doc_emb = np.mean(sbert_embeddings, axis=0)

        semantic_scores = cosine_similarity(
            sbert_embeddings,
            doc_emb.reshape(1, -1)
        ).flatten()

        semantic_scores = semantic_scores / (np.max(semantic_scores) + 1e-8)
    else:
        sbert_embeddings = None
        semantic_scores = np.zeros(len(sentences))

    # Combine
    alpha = config.get("alpha", 0.5)

    if config["use_tfidf"] and config["use_sbert"]:
        hybrid_scores = alpha * tfidf_scores + (1 - alpha) * semantic_scores
        embeddings = sbert_embeddings  
    elif config["use_tfidf"]:
        hybrid_scores = tfidf_scores
        embeddings = tfidf_embeddings  
    elif config["use_sbert"]:
        hybrid_scores = semantic_scores
        embeddings = sbert_embeddings  
    else:
        raise ValueError("At least one scoring must be enabled")

    return hybrid_scores, embeddings
def global_mmr(selected_indices, embeddings):
    final = []

    for idx in selected_indices:
        if not final:
            final.append(idx)
            continue

        sim = cosine_similarity(
            embeddings[idx].reshape(1, -1),
            embeddings[final]
        )[0]

        if max(sim) < 0.85:
            final.append(idx)

    return final  
def ralss_sentence_selection(sentences, roles, config, compression_ratio=0.25):

    if len(sentences) == 0:
        return []

    summary_limit = estimate_summary_length(sentences)

    # Hybrid scoring
    base_scores, embeddings = compute_hybrid_scores(
        sentences,
        config["scoring"]
    )

    weighted_scores = []
    for i in range(len(sentences)):

        role = roles[i]

        #  A1: Role Weights Ablation
        role_weight = ROLE_WEIGHTS.get(role, 0.2) if config["use_role_weights"] else 1.0

        #  A2: Position Weights Ablation
        pos_weight = position_weight(i, len(sentences)) if config["use_position_weights"] else 1.0

        length_bonus = 1 + (len(sentences[i].split()) / 100)

        score = base_scores[i] * role_weight * pos_weight * length_bonus
        weighted_scores.append(score)

    weighted_scores = np.array(weighted_scores)

    # Group by role
    role_groups = {}
    for i, role in enumerate(roles):
        role_groups.setdefault(role, []).append(i)

    selected_indices = []

    # --------------------------
    # Selection
    # --------------------------
    for role, idxs in role_groups.items():

        #  A4: Dynamic Budget Ablation
        if config["use_dynamic_budget"]:
            role_weight = ROLE_WEIGHTS.get(role, 0.2)
            role_budget = math.ceil(len(idxs) * role_weight * compression_ratio)
        else:
            role_budget = max(1, int(len(sentences) * compression_ratio / len(role_groups)))

        role_budget = max(role_budget, 1)

        role_sentences = [sentences[i] for i in idxs]
        role_scores = [weighted_scores[i] for i in idxs]
        role_embeddings = embeddings[idxs]

        # 🔹 A3: MMR Ablation
        if config["use_mmr"]:
            chosen = mmr_selection(
                role_sentences,
                role_scores,
                role_embeddings,
                role_budget
            )
        else:
            chosen = sorted(
                range(len(role_scores)),
                key=lambda x: role_scores[x],
                reverse=True
            )[:role_budget]

        selected_indices.extend([idxs[i] for i in chosen])

    #  Global MMR Ablation
    if config["use_global_mmr"]:
        selected_indices = global_mmr(selected_indices, embeddings)

    # Limit size
    if len(selected_indices) > summary_limit:
        ranked = sorted(
            [(weighted_scores[i], i) for i in selected_indices],
            reverse=True
        )
        selected_indices = [i for _, i in ranked[:summary_limit]]

    #  A5: Legal Order Ablation
    if config["use_legal_order"]:
        role_order = {r: i for i, r in enumerate(LEGAL_FLOW)}
        selected_indices.sort(
            key=lambda x: (role_order.get(roles[x], 999), x)
        )

    return selected_indices

# ==============================
# Ablation Variants
# ==============================

SCORING_ABLATIONS = {
    "TFIDF_ONLY": {
        "use_tfidf": True,
        "use_sbert": False
    },
    "SBERT_ONLY": {
        "use_tfidf": False,
        "use_sbert": True
    },
    "HYBRID_50_50": {
        "use_tfidf": True,
        "use_sbert": True,
        "alpha": 0.5
    },
    "HYBRID_80_20": {
        "use_tfidf": True,
        "use_sbert": True,
        "alpha": 0.8
    },
    "HYBRID_20_80": {
        "use_tfidf": True,
        "use_sbert": True,
        "alpha": 0.2
    },
    "HYBRID_35_65": {
        "use_tfidf": True,
        "use_sbert": True,
        "alpha": 0.35
    },
    "HYBRID_45_55": {
        "use_tfidf": True,
        "use_sbert": True,
        "alpha": 0.45
    },
    "HYBRID_55_45": {
        "use_tfidf": True,
        "use_sbert": True,
        "alpha": 0.55
    }
}

def build_structured_input(sentences, roles, selected_indices):
    structured = []

    for idx in selected_indices:
        role = roles[idx]
        sent = sentences[idx]
        structured.append(f"[{role}] {sent}")

    return " ".join(structured)
    
from transformers import pipeline

qa_model = pipeline(
    "question-answering",
    model="deepset/roberta-base-squad2",
    device=0 if torch.cuda.is_available() else -1
)

def refine_summary(chunk_summaries):

    all_sents = []
    for chunk in chunk_summaries:
        all_sents.extend(sent_tokenize(chunk))

    if len(all_sents) == 0:
        return ""

    embeddings = sbert_model.encode(all_sents)

    doc_emb = np.mean(embeddings, axis=0)

    scores = cosine_similarity(
        embeddings,
        doc_emb.reshape(1, -1)
    ).flatten()

    top_k = min(len(all_sents), 20)

    ranked = sorted(zip(scores, all_sents), reverse=True)

    selected = [s for _, s in ranked[:top_k]]

    return " ".join(selected)
    
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from nltk import sent_tokenize
def summac_chunked_fast(model, pred, judgement_chunks):

    pred_sents = sent_tokenize(pred)

    scores = []

    for chunk in judgement_chunks:
        for ps in pred_sents:

            score = model.score([chunk], [ps])["scores"][0]
            scores.append(score)

    return float(np.mean(scores))
import gc  

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks

def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary

results = {}

for variant_name, scoring_config in SCORING_ABLATIONS.items():

    config = {
        "use_role_weights": False,
        "use_position_weights": True,
        "use_mmr": False,
        "use_dynamic_budget": True,
        "use_legal_order": True,
        "use_global_mmr": False,
        "scoring": scoring_config
    }

    print(f"\n Running Scoring Variant: {variant_name}")


    factcc_scores, bart_scores = [], []
    rouge1_scores, rouge2_scores, rougel_scores, rougelsum_scores = [], [], [], []
    bleu_scores, meteor_scores, bert_scores = [], [], []
   

    for doc in tqdm(dataset, desc=variant_name):

        reference = " ".join(doc["headnote_sent"])

        judgement = " ".join(
            preprocess(s["sentence"])
            for s in doc["judgement_roles"]
            if isinstance(s, dict) and "sentence" in s
        )

        sentences = [s["sentence"] for s in doc["judgement_roles"]]
        roles = [s["role"] for s in doc["judgement_roles"]]

        selected_indices = ralss_sentence_selection(
            sentences,
            roles,
            config
        )
        # ranked_text = " ".join([sentences[i] for i in selected_indices])
        ranked_text = build_structured_input(sentences, roles, selected_indices)

        pred = generate_summary_chunked(
            ranked_text,
            tokenizer,
            model
        )

        # ROUGE
        r = rouge.compute(
            predictions=[pred],
            references=[reference]
        )
        rouge1_scores.append(r["rouge1"])
        rouge2_scores.append(r["rouge2"])
        rougel_scores.append(r["rougeL"])
        rougelsum_scores.append(r["rougeLsum"])
    
        bleu_doc = bleu.compute(
            predictions=[pred],
            references=[[reference]]
        )
        bleu_scores.append(bleu_doc["bleu"])
    
   
        meteor_doc = meteor.compute(
            predictions=[pred],
            references=[reference]
        )
        meteor_scores.append(meteor_doc["meteor"])
    
        bert_doc = bertscore.compute(
            predictions=[pred],
            references=[reference],
            lang="en"
        )
        bert_scores.append(bert_doc["f1"][0])
        
        bart_scores.append(
            bart_scorer.score([pred], [reference])[0]
        )
        # FactCC
        judgement_chunks = chunk_text(
            judgement,
            tokenizer,
            max_tokens=900,
            stride=150
        )
        factcc_scores.append(
            factcc_chunked_fast(pred, judgement_chunks)
        )
        
        del reference
        del judgement
        del sentences
        del roles
        del selected_indices
        del ranked_text
        del pred
        del judgement_chunks
        del bleu_doc
        del meteor_doc
        del bert_doc
        del r
        
        torch.cuda.empty_cache()
        gc.collect()
    print(f"ROUGE-1     ✅ {np.mean(rouge1_scores):.4f}")
    print(f"ROUGE-2     ✅ {np.mean(rouge2_scores):.4f}")
    print(f"ROUGE-L     ✅ {np.mean(rougel_scores):.4f}")
    print(f"ROUGE-LSUM  ✅ {np.mean(rougelsum_scores):.4f}")
    print(f"BLEU        ✅ {np.mean(bleu_scores):.4f}")
    print(f"METEOR      ✅ {np.mean(meteor_scores):.4f}")
    print(f"BERTScore   ✅ {np.mean(bert_scores):.4f}")
    print(f"BARTScore   ✅ {np.mean(bart_scores):.4f}")
    print(f"FactCC      ✅ {np.mean(factcc_scores):.4f}")

    results[variant_name] = {
        "ROUGE-1": np.mean(rouge1_scores),
        "ROUGE-2": np.mean(rouge2_scores),
        "ROUGE-L" : np.mean(rougel_scores),
        "ROUGE-LSUM": np.mean(rougelsum_scores),
        "BLEU": np.mean(bleu_scores),
        "METEOR" : np.mean(meteor_scores),
        "BERTSCORE": np.mean(bert_scores),
        "BARTSCORE": np.mean(bart_scores),
        "FactCC": np.mean(factcc_scores),
    }
    
print("\n📊 ABLATION RESULTS")

for k, v in results.items():
    print(f"\n{k}")
    for metric, score in v.items():
        print(f"{metric}: {score:.4f}")

# Length Generalizability

In [ ]:
import sys
import os
# 
# Add the BARTScore repo folder to Python path
sys.path.insert(0, os.path.abspath("/kaggle/working/BARTScore"))

# Test import
import bart_score
print("✅ BARTScore loaded successfully!")
import json
import math
import torch
import numpy as np
from tqdm import tqdm
import re
import evaluate
import torch.nn.functional as F
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer
from bart_score import BARTScorer
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.probability import FreqDist
import nltk
from nltk import sent_tokenize
from transformers import AutoModelForSeq2SeqLM
nltk.download("punkt")
nltk.download("stopwords")
import spacy

nlp = spacy.load("en_core_web_sm")
stop_words = set(stopwords.words("english"))
from sklearn.feature_extraction.text import CountVectorizer
from scipy.spatial.distance import jensenshannon

file_path = "/kaggle/input/datasets/pawank1999/rhetorical-role-dataset/rhetorical_role_dataset_500.json"

with open(file_path, "r") as f:
    dataset = json.load(f)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = AutoModelForSeq2SeqLM.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
).to(device)

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/input/datasets/pawank1999/distilbart-9500-output"
)

model.eval()
rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")
meteor = evaluate.load("meteor")
bertscore = evaluate.load("bertscore")
# BARTScore
bart_scorer = BARTScorer(
    device=device,
    checkpoint="facebook/bart-large-cnn"
)

def preprocess_text(sentences):
    processed_sentences = []
    for sent in sentences:
        words = [
            w.lower()
            for w in word_tokenize(sent)
            if w.isalpha() and w.lower() not in stop_words
        ]
        processed_sentences.append(" ".join(words))
    return sentences, processed_sentences


import torch
import numpy as np
from transformers import AutoTokenizer, AutoModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer_bert = AutoTokenizer.from_pretrained("nlpaueb/legal-bert-base-uncased")
model_bert = AutoModel.from_pretrained("nlpaueb/legal-bert-base-uncased").to(device)
model_bert.eval()
        
def legalbert_encode(sentences):
    embeddings = []

    for sent in sentences:
        inputs = tokenizer_bert(
            sent,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.no_grad():
            outputs = model_bert(**inputs)

        cls_embedding = outputs.last_hidden_state[:, 0, :]
        embeddings.append(cls_embedding.cpu().numpy()[0])

    return np.array(embeddings)
    
import random

def add_noise(sentences, roles, drop_prob=0.1):
    new_sent, new_roles = [], []
    for s, r in zip(sentences, roles):
        if np.random.rand() > drop_prob:
            new_sent.append(s)
            new_roles.append(r)
    return new_sent, new_roles


def shuffle_input(sentences, roles):
    combined = list(zip(sentences, roles))
    random.shuffle(combined)
    s, r = zip(*combined)
    return list(s), list(r)


# FactCC
factcc_model_name = "manueldeprada/FactCC"

factcc_tokenizer = AutoTokenizer.from_pretrained(factcc_model_name)
factcc_model = AutoModelForSequenceClassification.from_pretrained(
    factcc_model_name
).to(device)

factcc_model.eval()

def preprocess(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans

sbert_model = SentenceTransformer('all-mpnet-base-v2', device=device)


import networkx as nx
from sklearn.metrics.pairwise import cosine_similarity

def estimate_summary_length(sentences,
                            ratio=0.25,
                            min_len=8,
                            max_len=30):
    N = len(sentences)
    L = int(N * ratio)
    return max(min_len, min(L, max_len))

from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

import json
import math
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


# ==============================
# Legal Role Order
# ==============================

LEGAL_FLOW = [
    "PREAMBLE", "FAC", "ISSUE", "ARG_PETITIONER", "ARG_RESPONDENT",
    "RLC", "STA", "PRE_RELIED","PRE_NOT_RELIED", "ANALYSIS", "RATIO", "RPC"
]

# ==============================
# Role Importance Weights
# ==============================

ROLE_WEIGHTS = {
    "ANALYSIS": 1.0,
    "RATIO": 1.0,
    "RPC": 0.95,
    "FAC": 0.9,
    "ISSUE": 0.85,
    "RLC": 0.8,
    "PRE_RELIED": 0.7,
    "STA": 0.6,
    "ARG_PETITIONER": 0.5,
    "ARG_RESPONDENT": 0.5,
    "PREAMBLE": 0.4,
    "NONE": 0.2
}

# ==============================
# Position Weight
# ==============================

def position_weight(idx, total):
    """
    Early and late sentences get higher weight.
    """
    if total <= 1:
        return 1.0
    relative = idx / total
    if relative < 0.15:
        return 1.2
    elif relative > 0.85:
        return 1.15
    else:
        return 1.0

def mmr_selection(sentences, scores, embeddings, top_k, lambda_param=0.7):
    selected = []
    candidates = list(range(len(sentences)))
    while len(selected) < min(top_k, len(sentences)):
        mmr_scores = []
        for idx in candidates:
            relevance = scores[idx]
            redundancy = 0
            if selected:
                sim = cosine_similarity(
                    embeddings[idx].reshape(1, -1),
                    embeddings[selected]
                )[0]
                redundancy = max(sim)
            mmr_score = lambda_param * relevance - (1 - lambda_param) * redundancy
            mmr_scores.append((mmr_score, idx))
        mmr_scores.sort(reverse=True)
        best_idx = mmr_scores[0][1]
        selected.append(best_idx)
        candidates.remove(best_idx)
    return selected

def compute_hybrid_scores(sentences):
    # TF-IDF
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf_matrix = vectorizer.fit_transform(sentences)
    tfidf_embeddings = tfidf_matrix.toarray()
    tfidf_scores = np.sum(tfidf_embeddings, axis=1)

    # SBERT
    sbert_embeddings = sbert_model.encode(sentences)
    doc_emb = np.mean(sbert_embeddings, axis=0)
    semantic_scores = cosine_similarity(
        sbert_embeddings,
        doc_emb.reshape(1, -1)
    ).flatten()

    # Normalize
    tfidf_scores = tfidf_scores / (np.max(tfidf_scores) + 1e-8)
    semantic_scores = semantic_scores / (np.max(semantic_scores) + 1e-8)
    # Hybrid
    hybrid_scores = 0.65 * tfidf_scores + 0.35 * semantic_scores
    return hybrid_scores, tfidf_embeddings
    
def global_mmr(selected_indices, embeddings):
    final = []

    for idx in selected_indices:
        if not final:
            final.append(idx)
            continue

        sim = cosine_similarity(
            embeddings[idx].reshape(1, -1),
            embeddings[final]
        )[0]

        if max(sim) < 0.85:
            final.append(idx)

    return final  
def ralss_sentence_selection(sentences, roles, config, compression_ratio=0.25):

    if len(sentences) == 0:
        return []

    summary_limit = estimate_summary_length(sentences)

    # Hybrid scoring
    base_scores, embeddings = compute_hybrid_scores(sentences)

    weighted_scores = []
    for i in range(len(sentences)):

        role = roles[i]

        #  A1: Role Weights Ablation
        role_weight = ROLE_WEIGHTS.get(role, 0.2) if config["use_role_weights"] else 1.0

        # A2: Position Weights Ablation
        pos_weight = position_weight(i, len(sentences)) if config["use_position_weights"] else 1.0

        length_bonus = 1 + (len(sentences[i].split()) / 100)

        score = base_scores[i] * role_weight * pos_weight * length_bonus
        weighted_scores.append(score)

    weighted_scores = np.array(weighted_scores)

    # Group by role
    role_groups = {}
    for i, role in enumerate(roles):
        role_groups.setdefault(role, []).append(i)

    selected_indices = []

    # --------------------------
    # Selection
    # --------------------------
    for role, idxs in role_groups.items():

        #  A4: Dynamic Budget Ablation
        if config["use_dynamic_budget"]:
            role_weight = ROLE_WEIGHTS.get(role, 0.2)
            role_budget = math.ceil(len(idxs) * role_weight * compression_ratio)
        else:
            role_budget = max(1, int(len(sentences) * compression_ratio / len(role_groups)))

        role_budget = max(role_budget, 1)

        role_sentences = [sentences[i] for i in idxs]
        role_scores = [weighted_scores[i] for i in idxs]
        role_embeddings = embeddings[idxs]

        #  A3: MMR Ablation
        if config["use_mmr"]:
            chosen = mmr_selection(
                role_sentences,
                role_scores,
                role_embeddings,
                role_budget
            )
        else:
            chosen = sorted(
                range(len(role_scores)),
                key=lambda x: role_scores[x],
                reverse=True
            )[:role_budget]

        selected_indices.extend([idxs[i] for i in chosen])

    #  Global MMR Ablation
    if config["use_global_mmr"]:
        selected_indices = global_mmr(selected_indices, embeddings)

    # Limit size
    if len(selected_indices) > summary_limit:
        ranked = sorted(
            [(weighted_scores[i], i) for i in selected_indices],
            reverse=True
        )
        selected_indices = [i for _, i in ranked[:summary_limit]]

    #  A5: Legal Order Ablation
    if config["use_legal_order"]:
        role_order = {r: i for i, r in enumerate(LEGAL_FLOW)}
        selected_indices.sort(
            key=lambda x: (role_order.get(roles[x], 999), x)
        )

    return selected_indices


def get_length_bucket(num_sentences):
    if num_sentences < 52:
        return "short"
    elif num_sentences <= 131:
        return "medium"
    else:
        return "long"
        
def build_structured_input(sentences, roles, selected_indices):
    structured = []

    for idx in selected_indices:
        role = roles[idx]
        sent = sentences[idx]
        structured.append(f"[{role}] {sent}")

    return " ".join(structured)
    
from transformers import pipeline
    
def factcc_chunked_fast(pred, judgement_chunks):
    chunk_scores = []

    for chunk in judgement_chunks:
        inputs = factcc_tokenizer(
            pred,
            chunk,
            return_tensors="pt",
            truncation=True,
            padding=True,
            max_length=512
        ).to(device)

        with torch.inference_mode():
            logits = factcc_model(**inputs).logits
            probs = F.softmax(logits, dim=-1)

        chunk_scores.append(probs[0, 1].item())

    return float(np.mean(chunk_scores))

from nltk import sent_tokenize
def summac_chunked_fast(model, pred, judgement_chunks):

    pred_sents = sent_tokenize(pred)

    scores = []

    for chunk in judgement_chunks:
        for ps in pred_sents:

            score = model.score([chunk], [ps])["scores"][0]
            scores.append(score)

    return float(np.mean(scores))
import gc  

def chunk_text(text, tokenizer, max_tokens=900, stride=50):
    """
    Splits text into overlapping token chunks
    """
    tokens = tokenizer.encode(text, add_special_tokens=False)

    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk_text = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        chunks.append(chunk_text)
        start = end - stride  # overlap

    return chunks

def generate_summary_chunked(text, tokenizer, model):
    """
    Chunk → summarize → merge
    """
    chunks = chunk_text(text, tokenizer)

    chunk_summaries = []

    for chunk in chunks:
        inputs = tokenizer(
            chunk,
            return_tensors="pt",
            truncation=True,
            max_length=1024
        ).to(device)

        with torch.no_grad():
            summary_ids = model.generate(
                **inputs,
                num_beams=4,
                no_repeat_ngram_size=3,
                length_penalty=2.0,
                min_new_tokens=150,
                max_new_tokens=350,
                early_stopping=True
            )

        summary = tokenizer.decode(
            summary_ids[0],
            skip_special_tokens=True,
            clean_up_tokenization_spaces=False
        )

        chunk_summaries.append(summary)

    # Merge chunk summaries
    merged_summary = " ".join(chunk_summaries)
    return merged_summary

def evaluate_metrics(pred, reference, judgement_chunks):

    r = rouge.compute(
        predictions=[pred],
        references=[reference]
    )

    bleu_score = bleu.compute(
        predictions=[pred],
        references=[[reference]]
    )["bleu"]

    meteor_score = meteor.compute(
        predictions=[pred],
        references=[reference]
    )["meteor"]

    bert_score_val = bertscore.compute(
        predictions=[pred],
        references=[reference],
        lang="en"
    )["f1"][0]

    factcc_val = factcc_chunked_fast(pred, judgement_chunks)

    return {
        "rouge1": r["rouge1"],
        "rouge2": r["rouge2"],
        "rougeL": r["rougeL"],
        "bleu": bleu_score,
        "meteor": meteor_score,
        "bertscore": bert_score_val,
        "factcc": factcc_val
    }

length_results = {
    "short": [],
    "medium": [],
    "long": []
}

MAIN_CONFIG = {
    "use_role_weights": False,
    "use_position_weights": True,
    "use_mmr": False,
    "use_dynamic_budget": True,
    "use_legal_order": True,
    "use_global_mmr": False
}
from tqdm import tqdm
import numpy as np
import gc

for doc in tqdm(dataset, desc="Length Generalization"):

    # -----------------------
    # Reference
    # -----------------------
    reference = " ".join(doc["headnote_sent"])

    judgement = " ".join(
        preprocess(s["sentence"])
        for s in doc["judgement_roles"]
        if isinstance(s, dict) and "sentence" in s
    )

    sentences = [s["sentence"] for s in doc["judgement_roles"]]
    roles = [s["role"] for s in doc["judgement_roles"]]

    #  Assign bucket (UPDATED)
    bucket = get_length_bucket(len(sentences))

    selected_indices = ralss_sentence_selection(
        sentences,
        roles,
        config=MAIN_CONFIG
    )

    ranked_text = build_structured_input(sentences, roles, selected_indices)

    pred = generate_summary_chunked(
        ranked_text,
        tokenizer,
        model
    )

    judgement_chunks = chunk_text(
        judgement,
        tokenizer,
        max_tokens=900,
        stride=150
    )

    scores = evaluate_metrics(pred, reference, judgement_chunks)

    length_results[bucket].append(scores)

    # cleanup
    del reference, judgement, sentences, roles
    del selected_indices, ranked_text, pred, judgement_chunks
    torch.cuda.empty_cache()
    gc.collect()
    
def aggregate_scores(scores_list):
    return {
        key: np.mean([s[key] for s in scores_list])
        for key in scores_list[0]
    }

final_length_results = {}

for bucket in ["short", "medium", "long"]:
    if len(length_results[bucket]) > 0:
        final_length_results[bucket] = aggregate_scores(length_results[bucket])

print("\n📊 LENGTH GENERALIZATION RESULTS\n")

for bucket in ["short", "medium", "long"]:
    if bucket in final_length_results:
        res = final_length_results[bucket]

        print(f"--- {bucket.upper()} DOCUMENTS ---")
        print(f"ROUGE-1   ✅ {res['rouge1']:.4f}")
        print(f"ROUGE-2   ✅ {res['rouge2']:.4f}")
        print(f"ROUGE-L   ✅ {res['rougeL']:.4f}")
        print(f"BLEU      ✅ {res['bleu']:.4f}")
        print(f"METEOR    ✅ {res['meteor']:.4f}")
        print(f"BERTScore ✅ {res['bertscore']:.4f}")
        print(f"FactCC    ✅ {res['factcc']:.4f}")
        print()

# --- SHORT DOCUMENTS ---
# ROUGE-1   ✅ 0.5803
# ROUGE-2   ✅ 0.3538
# ROUGE-L   ✅ 0.3388
# BLEU      ✅ 0.2311
# METEOR    ✅ 0.3768
# BERTScore ✅ 0.8725
# FactCC    ✅ 0.9854

# --- MEDIUM DOCUMENTS ---
# ROUGE-1   ✅ 0.6026
# ROUGE-2   ✅ 0.3449
# ROUGE-L   ✅ 0.3084
# BLEU      ✅ 0.2338
# METEOR    ✅ 0.3710
# BERTScore ✅ 0.8605
# FactCC    ✅ 0.9871

# --- LONG DOCUMENTS ---
# ROUGE-1   ✅ 0.5704
# ROUGE-2   ✅ 0.2970
# ROUGE-L   ✅ 0.2583
# BLEU      ✅ 0.1874
# METEOR    ✅ 0.3344
# BERTScore ✅ 0.8489
# FactCC    ✅ 0.9796